In [1]:
!pip install transformers datasets evaluate tqdm scikit-learn pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.9.0.13 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn

In [ ]:
import json
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from time import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List


device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", trust_remote_code=True)
train_ds, test_ds = ds['train'], ds['test']

MODEL_NAME = 'dnagpt/gpt2_gene_v1'
PATH_TO_SAVE_OUTPUTS = '.'
BATCH_SIZE = 16

params_logreg = {'max_iter': 1000, 'random_state': 42}

def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class EmbeddingExtractor:
    """
    Извлекает эмбеддинги из последнего скрытого слоя HF CausalLM
    с mean-pooling по длине.
    """
    def __init__(self, model_name_or_path: str, device: str = None, seed: int = 42):
        set_seed(seed)
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
        self.model = (
            AutoModelForCausalLM.from_pretrained(
                model_name_or_path,
                output_hidden_states=True,
                return_dict=True
            )
            .to(self.device)
        )
        self.model.eval()

    def extract_embeddings(self, seqs: List[str], batch_size: int = 8):
        """
        Принимает список последовательностей (строк) и возвращает
        numpy.ndarray формы (N, H) — эмбеддинги mean-pooled
        из последнего скрытого слоя.
        """
        all_embeddings = []
        with torch.no_grad():
            for i in tqdm(range(0, len(seqs), batch_size), desc="Extracting embeddings"):
                batch = seqs[i : i + batch_size]
                enc = self.tokenizer(
                    batch,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    add_special_tokens=True
                )
                input_ids = enc.input_ids.to(self.device)
                attention_mask = enc.attention_mask.to(self.device)
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True
                )
                hidden_states = outputs.hidden_states[-1]  # (B, L, H)
                mask = attention_mask.unsqueeze(-1)       # (B, L, 1)
                summed = (hidden_states * mask).sum(dim=1) # (B, H)
                lengths = mask.sum(dim=1).clamp(min=1)     # (B, 1)
                emb = (summed / lengths).cpu().numpy()    # (B, H)
                all_embeddings.append(emb)
        return np.vstack(all_embeddings)


extractor = EmbeddingExtractor(MODEL_NAME, device=device)

baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task'] == t)
    te = test_ds.filter(lambda x, t=task: x['task'] == t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])

    
    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**params_logreg)
    
    Xf = X_tr.reshape(-1,1) if X_tr.ndim==1 or X_tr.shape[1]==1 else X_tr
    Xt = X_te.reshape(-1,1) if X_te.ndim==1 or X_te.shape[1]==1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    
    with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnahlm_task-{task}_baseline.json','w') as f:
        json.dump(baseline, f, indent=4)


def few_shot(train, test, ks=(1,5,10,20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task'] == t)
        te = test.filter(lambda x, t=task: x['task'] == t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])

        
        X_te = extractor.extract_embeddings(seqs_te, batch_size=16)
        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=16)
                y_k = y_tr[idxs]

                clf = LogisticRegression(**params_logreg)
                Xf = X_k.reshape(-1,1) if X_k.ndim==1 or X_k.shape[1]==1 else X_k
                Xt = X_te.reshape(-1,1) if X_te.ndim==1 or X_te.shape[1]==1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))

            res[task][k] = {
                'accuracy': float(np.mean(accs)),
                'f1_score': float(np.mean(f1s))
            }
            with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnahlm_task-{task}_k-{k}.json','w') as f:
                json.dump(res, f, indent=4)
    return res

results_kshot = few_shot(train_ds, test_ds)

output = {'full': baseline, 'kshot': results_kshot, 'params': params_logreg}
        
with open(F'{PATH_TO_SAVE_OUTPUTS}/results_dnahlm.json','w') as f:
    json.dump(output, f, indent=4)

print("All benchmarks completed. Results saved to results_dnahlm.json")


Extracting embeddings:  84%|████████▍ | 1317/1563 [02:02<00:23, 10.68it/s]

Extracting embeddings: 100%|██████████| 1563/1563 [02:25<00:00, 10.76it/s]

Extracting embeddings: 100%|██████████| 1623/1623 [02:30<00:00, 10.77it/s]

Baseline:  50%|█████     | 9/18 [24:37<22:37, 150.88s/it]

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48797 [00:00<?, ? examples/s]


Extracting embeddings: 100%|██████████| 345/345 [00:21<00:00, 16.39it/s]

Baseline:  56%|█████▌    | 10/18 [25:07<15:08, 113.52s/it]

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]

...


Extracting embeddings: 100%|██████████| 25/25 [00:01<00:00, 24.28it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 75.82it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 53.33it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 63.42it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 57.86it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 60.06it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 21.39it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 23.39it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 23.86it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 26.41it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 26.19it/s]

Extracting embeddings: 100%|██████████| 2/2 [00:00<00:00, 24.53it/s]

Extracting embeddings: 100%|██████████| 2/2 [00:00<00:00, 23.43it/s]

Extracting embeddings: 100%|██████████| 2/2 [00:00<00:00, 23.43it/s]

Extracting embedd

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/48797 [00:00<?, ? examples/s]


Extracting embeddings: 100%|██████████| 370/370 [00:23<00:00, 16.05it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 73.74it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 45.46it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 46.08it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 50.67it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 44.87it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 21.51it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 24.09it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 23.07it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 20.02it/s]

Extracting embeddings: 100%|██████████| 1/1 [00:00<00:00, 19.25it/s]

Extracting embeddings: 100%|██████████| 2/2 [00:00<00:00, 22.27it/s]

Extracting embeddings: 100%|██████████| 2/2 [00:00<00:00, 23.10it/s]

Extracting embeddings: 100%|██████████| 2/2 [00:00<00:00, 22.40it/s]

Extracting embe

All benchmarks completed. Results saved to results_dnahlm.json
